In [ ]:
# install packages required

%pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
%pip install transformers
%pip install datasets
%pip install scikit-learn
%pip install 'accelerate>=1.1.0'
%pip install time
%pip install evaluate

In [ ]:
# loading packages


import duckdb
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from datasets import Dataset
from transformers import TrainingArguments, Trainer
import transformers
import accelerate
from transformers import BertForSequenceClassification
import time
import torch
import numpy as np
import evaluate
from transformers import BertTokenizer, BertForSequenceClassification
import pickle
import os
import s3fs


# torch installation needed (pip pip install -U accelerate) ou pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
# transformers installation needed (pip install transformers)
# datasets installation needed (pip install datasets)
# pip install scikit-learn
# pip install 'accelerate>=1.1.0'

In [ ]:
# loading the dataset

con = duckdb.connect(database=":memory:")
con.execute("INSTALL httpfs;")
con.execute("LOAD httpfs;")

query = "SELECT * FROM read_parquet('https://minio.lab.sspcloud.fr/projet-formation/diffusion/funathon/2026/project2/generation_None_temp08.parquet')"
df = con.sql(query).df()

In [ ]:
df.info()

In [ ]:
# create NAF
nomenclature = df[['code', 'name']].drop_duplicates()

In [ ]:
#import the NACE


con = duckdb.connect(database=":memory:")

con.execute("INSTALL httpfs;")
con.execute("LOAD httpfs;")

path_nace = 'https://minio.lab.sspcloud.fr/projet-formation/diffusion/funathon/2026/project2/NACE_Rev2.1_Structure_Explanatory_Notes_EN.tsv'
query_definition = f"SELECT * FROM read_csv('{path_nace}')"
table = con.execute(query_definition).to_arrow_table()
nace = table.to_pylist()
nace[1]

nace2 = pd.DataFrame(nace)


In [ ]:
nace2.head()

In [ ]:
nace_filtered = nace2[nace2["CODE"].astype(str).str.len() == 5]
nace_filtered= nace_filtered[['CODE', 'Includes', 'IncludesAlso', 'Excludes' ]]
nace_filtered["Includes"] = nace_filtered["Includes"].str.replace("\\n", "", regex=False)
nace_filtered["IncludesAlso"] = nace_filtered["IncludesAlso"].str.replace("\\n", "", regex=False)
nace_filtered["Excludes"] = nace_filtered["Excludes"].str.replace("\\n", "", regex=False)

In [ ]:
nace_filtered.head()
#nomenclature.head()

In [ ]:
# label encoding
le = LabelEncoder()
df['target'] = le.fit_transform(df['code'])

num_labels = df['target'].nunique()
print(num_labels)

In [ ]:
#### TEST ####

from sklearn.utils.class_weight import compute_class_weight
import torch

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(df["target"]),
    y=df["target"]
)

class_weights = torch.tensor(class_weights, dtype=torch.float)
# todo - augmenter la taille du batch pour tester 

In [ ]:
# create train and test dataset
train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df['target'],  # class equilibtrate
    random_state=42
)

In [ ]:
# tokenize

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

def tokenize(batch):
    return tokenizer(
        batch['label'],
        padding='max_length',
        truncation=True,
        max_length=128
    )


In [ ]:

# conversion in datasets
train_dataset = Dataset.from_pandas(train_df[['label', 'target']])
test_dataset = Dataset.from_pandas(test_df[['label', 'target']])

train_dataset = train_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

train_dataset = train_dataset.rename_column("target", "labels")
test_dataset = test_dataset.rename_column("target", "labels")

train_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])
test_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])


# temporary dataset reduced
train_dataset = train_dataset.shuffle().select(range(20))
test_dataset = test_dataset.shuffle().select(range(4))


In [ ]:

# charge bert model

model = BertForSequenceClassification.from_pretrained(
    'bert-base-uncased',
    num_labels=num_labels
)

In [ ]:
# trainning

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    logging_dir="./logs",
    disable_tqdm=False,   # 🔥 important
    report_to="none"     
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset
)

start = time.time()

trainer.train()

end = time.time()

print(f"Training time : {end - start:.2f} secondes")

### saving model, tokenizer, encoder and nace
# dossier local temporaire
LOCAL_DIR = "./model_nace"
os.makedirs(LOCAL_DIR, exist_ok=True)

# save local
trainer.save_model("./model_nace")
tokenizer.save_pretrained("./model_nace")
with open("./model_nace/label_encoder.pkl", "wb") as f:
    pickle.dump(le, f)

nomenclature.to_csv("./model_nace/nomenclature.csv", index=False)


# Create filesystem object
S3_ENDPOINT_URL = "https://" + os.environ["AWS_S3_ENDPOINT"]
fs = s3fs.S3FileSystem(client_kwargs={'endpoint_url': S3_ENDPOINT_URL})

trainer.save_model(LOCAL_DIR)
tokenizer.save_pretrained(LOCAL_DIR)

with open(f"{LOCAL_DIR}/label_encoder.pkl", "wb") as f:
    pickle.dump(le, f)

nomenclature.to_csv(f"{LOCAL_DIR}/nomenclature.csv", index=False)

# -------------------
# upload vers bucket
# -------------------

BUCKET_OUT = "thierry57"

files_to_upload = [
    "config.json",
    "tokenizer_config.json",
    "tokenizer.json",
    "model.safetensors",         # ou pytorch_model.bin
    "label_encoder.pkl",
    "nomenclature.csv",
    "training_args.bin"
]

for file_name in files_to_upload:

    local_path = f"{LOCAL_DIR}/{file_name}"
    s3_path = f"{BUCKET_OUT}/model_nace/{file_name}"

    if os.path.exists(local_path):

        with open(local_path, "rb") as f_in:
            with fs.open(s3_path, "wb") as f_out:
                f_out.write(f_in.read())

        print(f"Uploaded: {s3_path}")


In [ ]:
results = trainer.evaluate()
print(results)

In [ ]:
metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return metric.compute(predictions=preds, references=labels)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

In [ ]:
trainer.evaluate()

In [ ]:
predictions = trainer.predict(test_dataset)

In [ ]:
y_pred = np.argmax(predictions.predictions, axis=1)
y_true = predictions.label_ids

In [ ]:
from sklearn.metrics import accuracy_score

print("Accuracy:", accuracy_score(y_true, y_pred))

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_true, y_pred))

In [ ]:
y_pred_code = le.inverse_transform(y_pred)
y_true_code = le.inverse_transform(y_true)

In [ ]:
df_results = test_df.copy().reset_index(drop=True)

df_results["true_code"] = y_true_code
df_results["pred_code"] = y_pred_code

In [ ]:
df_results = df_results.merge(
    nomenclature.rename(columns={"code": "true_code", "name": "true_name"}),
    on="true_code",
    how="left"
)

In [ ]:
df_results = df_results.merge(
    nomenclature.rename(columns={"code": "pred_code", "name": "pred_name"}),
    on="pred_code",
    how="left"
)
df_results

In [ ]:
df_errors = df_results[df_results["true_code"] != df_results["pred_code"]]
df_errors.head(20)

In [ ]:

# charger depuis ton dossier sauvegardé
model = BertForSequenceClassification.from_pretrained("./model_nace")
tokenizer = BertTokenizer.from_pretrained("./model_nace")

# device (GPU si dispo)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

def predict_label(text, model, tokenizer, le, nomenclature, device):
    # tokenisation
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    # envoyer sur device
    inputs = {k: v.to(device) for k, v in inputs.items()}

    # prédiction
    with torch.no_grad():
        outputs = model(**inputs)

    logits = outputs.logits
    pred_class_id = torch.argmax(logits, dim=1).cpu().numpy()[0]

    # décodage du label
    pred_code = le.inverse_transform([pred_class_id])[0]

    # récupérer le libellé associé
    pred_name = nomenclature.loc[
        nomenclature["code"] == pred_code, "name"
    ].values[0]

    return {
        "text": text,
        "pred_class_id": pred_class_id,
        "pred_code": pred_code,
        "pred_name": pred_name
    }

In [ ]:
text = "someone who grows lettuce"

result = predict_label(text, model, tokenizer, le, nomenclature, device)

print("Texte :", result["text"])
print("Classe prédite :", result["pred_class_id"])
print("Code prédit :", result["pred_code"])
print("Libellé prédit :", result["pred_name"])

In [ ]:

nomenclature.to_csv("s3/thierry57/nomenclature.csv", index=False)
nace2.to_csv("s3/thierry57/nace.csv", index=False)


In [ ]:
import os
import s3fs
# Create filesystem object
S3_ENDPOINT_URL = "https://" + os.environ["AWS_S3_ENDPOINT"]
fs = s3fs.S3FileSystem(client_kwargs={'endpoint_url': S3_ENDPOINT_URL})


In [ ]:
fs.ls("thierry57")

In [ ]:
# read file

BUCKET = "thierry57"
FILE_KEY_S3 = "naf_en_fr.csv"
FILE_PATH_S3 = BUCKET + "/" + FILE_KEY_S3

with fs.open(FILE_PATH_S3, mode="rb") as file_in:
    df_bpe = pd.read_csv(file_in, sep=";")

In [ ]:
#write file

BUCKET_OUT = "thierry57"
FILE_KEY_OUT_S3 = "nace_filtered.csv"
FILE_PATH_OUT_S3 = BUCKET_OUT + "/" + FILE_KEY_OUT_S3

with fs.open(FILE_PATH_OUT_S3, 'w') as file_out:
    nace_filtered.to_csv(file_out)